# Training "hey gemma" wake-word model

Sprint R3 of the `gemma4_agent` project. Trains a custom openWakeWord model on the operator's annotated audio.

**Time**: ~30-60 min on a free Colab T4 GPU.
**Inputs**: `wake_dataset.zip` (produced by `scripts/build_wake_dataset.py`, uploaded in Cell 3).
**Output**: `hey_gemma.onnx` (downloaded in Cell 7).

After downloading the ONNX file, drop it into the repo at:

```
gemma4_agent/voice/models/oww/hey_gemma.onnx
```

Then run `python scripts/wake_backend_audit.py` and `python scripts/bench.py` to measure the new recall + precision. Acceptance bar (per the project memory `feedback_wake_acceptance_bar`): **recall >= 0.70 AND precision >= 0.80**. Custom models that hit recall but tank precision (lots of phantom fires) are **not shippable** -- false positives interrupt the operator and are worse UX than the Vosk baseline (recall 0.057, precision 0.857).

In [ ]:
# Cell 2: install + setup. ~30s on a fresh runtime.
!pip install -q openwakeword onnxruntime
import openwakeword
print('openwakeword module loaded')
# Print available training entrypoints so we know which API
# version we are on. The exact symbol names can drift between
# openwakeword releases; Cell 5 adapts to what's available here.
import inspect
try:
    from openwakeword import train as oww_train
    print('openwakeword.train exports:', [n for n in dir(oww_train) if not n.startswith('_')])
except Exception as exc:
    print('openwakeword.train not importable:', exc)

In [ ]:
# Cell 3: upload the dataset zip produced by
# scripts/build_wake_dataset.py on your local machine.
from google.colab import files
print('Upload wake_dataset.zip (produced by build_wake_dataset.py):')
uploaded = files.upload()
!unzip -q wake_dataset.zip
!ls wake_dataset/

In [ ]:
# Cell 4: validate the dataset before kicking off training.
# Cheap fail-fast: if you accidentally uploaded the wrong zip
# or skipped augmentation, this stops you here, not 30 min in.
import json
with open('wake_dataset/manifest.json') as f:
    manifest = json.load(f)
print(json.dumps(manifest, indent=2))
assert manifest['positives']['raw_count'] >= 100, (
    f"Need >= 100 raw positives; manifest reports {manifest['positives']['raw_count']}."
)
assert manifest['hard_negatives']['count'] >= 50, (
    f"Need >= 50 hard negatives; manifest reports {manifest['hard_negatives']['count']}."
)

In [ ]:
# Cell 5: configure training. The openwakeword.train API has
# changed across versions; this cell uses introspection to find
# the right entrypoint. If your version exposes a different name,
# the printout in Cell 2 lists the actual symbols available --
# adapt the imports below to match.
from pathlib import Path
from openwakeword import train as oww_train

POSITIVE_DIR = Path('wake_dataset/positive')
POSITIVE_AUG_DIR = Path('wake_dataset/positive_aug')
NEGATIVE_DIR = Path('wake_dataset/negative_hard')

config = {
    'wake_word': 'hey_gemma',
    'positive_clips_dir': str(POSITIVE_DIR),
    'positive_augmented_dir': str(POSITIVE_AUG_DIR) if POSITIVE_AUG_DIR.exists() else None,
    'hard_negative_clips_dir': str(NEGATIVE_DIR),
    'use_default_negatives': True,
    'target_metric': 'f1',
    'target_metric_floor': 0.80,
    'training_steps': 10000,
    'batch_size': 64,
    'early_stopping': True,
    'early_stopping_patience': 5,
}
print('Config:')
print(json.dumps(config, indent=2))

# Pick the training entrypoint defensively. If your
# openwakeword version uses a different name, replace this with
# the correct one (Cell 2's printout lists the candidates).
if hasattr(oww_train, 'train_custom_model_pipeline'):
    train_fn = oww_train.train_custom_model_pipeline
elif hasattr(oww_train, 'train_custom_model'):
    train_fn = oww_train.train_custom_model
else:
    raise RuntimeError(
        'No known training entrypoint in openwakeword.train. '
        f"Available: {[n for n in dir(oww_train) if not n.startswith('_')]}"
    )
print('Using training entrypoint:', train_fn.__name__)

In [ ]:
# Cell 6: run training. ~30-60 min on a T4. Watch the metrics
# stream; you want f1 climbing toward 0.80+ on the held-out
# split before time runs out.
result = train_fn(**config)
print('Training done.')
print('Result:', result if isinstance(result, (str, dict, list)) else type(result).__name__)

In [ ]:
# Cell 7: download the trained .onnx for drop-in to the repo.
# The training pipeline produces a file under ./models/ or
# ./hey_gemma.onnx depending on the version; we search both.
import glob
candidates = (
    glob.glob('hey_gemma.onnx')
    + glob.glob('models/hey_gemma*.onnx')
    + glob.glob('**/hey_gemma*.onnx', recursive=True)
)
if not candidates:
    raise FileNotFoundError('No hey_gemma*.onnx found after training; check Cell 6 output.')
src = candidates[0]
print(f'Downloading {src} (rename target file: hey_gemma.onnx)')
import shutil
shutil.copyfile(src, 'hey_gemma.onnx')
from google.colab import files
files.download('hey_gemma.onnx')
print('Drop this file at: gemma4_agent/voice/models/oww/hey_gemma.onnx')

## After downloading

1. Move `hey_gemma.onnx` to `gemma4_agent/voice/models/oww/hey_gemma.onnx` in the repo.
2. Verify the file landed: `python -c "from pathlib import Path; p = Path('gemma4_agent/voice/models/oww/hey_gemma.onnx'); print(f'exists={p.exists()}, size={p.stat().st_size if p.exists() else 0}')"`
3. Benchmark per-backend recall/precision/F1: `python scripts/wake_backend_audit.py`
4. Re-measure the production-readiness bench: `python scripts/bench.py`

**Acceptance bar** (per the project's wake-acceptance memory and the Vosk baseline characterization):

- `recall >= 0.70` (12x Vosk's 0.057)
- `precision >= 0.80` (don't regress below Vosk's 0.857 floor)

If either threshold fails, see Sprint R3 follow-ups:

- **Low recall**: record 30-50 more positive samples in a different session (varied mic/room/day) and re-run R3.
- **Low precision**: harder negative mining (TTS-generated phonetic neighbours: "ay gema", "hey jefe", "tema", etc.).

Either is a small follow-up sprint, not a redo of R3.